# Analisis de resenas wise.com en Trustpilot

**Empresa:** Wise
**Sector:** Money and Insurance

### Objetivos
1. Las resenas son positivas o negativas? Y en la competencia?
2. Que temas tratan? Y en la competencia?
3. Por topic, el sentimiento es positivo o negativo? Donde somos mejores o peores?
4. Areas de mejora.

## PASO 1 - Librerias y configuracion

In [ ]:
# pip install pandas numpy matplotlib seaborn transformers torch bertopic umap-learn hdbscan scikit-learn wordcloud

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

CSV_PATH = 'trustpilot-reviews-123k.csv'  # ajusta la ruta si hace falta
TARGET   = 'wise.com'
CATEGORY = 'Money & Insurance'

def stars_to_label(s):
    if s <= 2: return 'Negativo'
    if s == 3: return 'Neutro'
    return 'Positivo'

PALETTE = {'Positivo': '#2ecc71', 'Neutro': '#f39c12', 'Negativo': '#e74c3c'}
print('OK')

## PASO 2 - Carga y exploracion inicial

In [ ]:
df_all = pd.read_csv(CSV_PATH)
print(f'Shape: {df_all.shape}')
df_all.head(3)

In [ ]:
df_sector = df_all[df_all['category'] == CATEGORY].copy()
df_wise   = df_all[df_all['company'] == TARGET].copy()
n_emp = df_sector['company'].nunique()
print(f'Sector: {len(df_sector)} resenas, {n_emp} empresas')
print(f'Wise: {len(df_wise)} resenas')
print(df_wise['stars'].value_counts().sort_index())

## PASO 3 - Limpieza de texto

In [ ]:
df_wise['review'].sample(5, random_state=42).tolist()

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def combine_text(row):
    t = str(row['title'])  if pd.notna(row.get('title'))  else ''
    r = str(row['review']) if pd.notna(row.get('review')) else ''
    return clean_text(t + ' ' + r)

df_wise['text_clean']   = df_wise.apply(combine_text, axis=1)
df_sector['text_clean'] = df_sector.apply(combine_text, axis=1)
print(df_wise['text_clean'].iloc[0])
n_empty = (df_wise['text_clean'] == '').sum()
print(f'Textos vacios en Wise: {n_empty}')

## PASO 4 - Analisis de sentimiento

Modelo: `nlptown/bert-base-multilingual-uncased-sentiment`

1-5 estrellas por resena. 1-2=Negativo | 3=Neutro | 4-5=Positivo

In [ ]:
sentiment_pipe = pipeline(
    'text-classification',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    truncation=True, max_length=512
)
print('Modelo cargado')

In [ ]:
def predict_sentiment(texts, pipe, batch_size=32):
    texts_trunc = [t[:512] if t else 'no review' for t in texts]
    preds = pipe(texts_trunc, batch_size=batch_size)
    out = []
    for p in preds:
        s = int(p['label'].split()[0])
        out.append({'stars_pred': s, 'sentiment': stars_to_label(s)})
    return out

print('Calculando sentimiento Wise...')
wise_sent = predict_sentiment(df_wise['text_clean'].tolist(), sentiment_pipe)
df_wise[['stars_pred', 'sentiment']] = pd.DataFrame(wise_sent)
print('Listo')
df_wise['sentiment'].value_counts()

In [ ]:
print('Calculando sentimiento sector (puede tardar unos minutos)...')
sector_sent = predict_sentiment(df_sector['text_clean'].tolist(), sentiment_pipe)
df_sector[['stars_pred', 'sentiment']] = pd.DataFrame(sector_sent)
print('Listo')

### 4.1 Sentimiento global de Wise

In [ ]:
order = ['Positivo', 'Neutro', 'Negativo']
wise_counts = df_wise['sentiment'].value_counts().reindex(order, fill_value=0)
wise_pct    = (wise_counts / wise_counts.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(order, wise_pct.values, color=[PALETTE[o] for o in order], edgecolor='white', linewidth=1.5)
for bar, pct in zip(bars, wise_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_title('Sentimiento global - Wise.com', fontsize=14, fontweight='bold')
ax.set_ylabel('% resenas')
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
sns.despine()
plt.tight_layout()
plt.savefig('wise_sentimiento_global.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Comparativa con la competencia

In [ ]:
sent_comp = (df_sector.groupby('company')['sentiment']
             .value_counts(normalize=True).mul(100).rename('pct').reset_index())
pos_comp  = sent_comp[sent_comp['sentiment'] == 'Positivo'].sort_values('pct', ascending=True)

colors = ['#e74c3c' if c == TARGET else '#3498db' for c in pos_comp['company']]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(pos_comp['company'], pos_comp['pct'], color=colors)
wise_val = pos_comp[pos_comp['company'] == TARGET]['pct'].values
if len(wise_val):
    ax.axvline(wise_val[0], color='#e74c3c', linestyle='--', linewidth=1.5, label='Wise')
ax.set_title('% Resenas Positivas - Sector Money and Insurance', fontsize=13, fontweight='bold')
ax.set_xlabel('% Positivo')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('wise_sentimiento_competencia.png', dpi=150, bbox_inches='tight')
plt.show()

## PASO 5 - Topics con BERTopic

Entrenamos sobre todo el sector (~3000 resenas) para mayor robustez. Luego filtramos los topics de Wise.

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

docs = df_sector['text_clean'].tolist()

umap_model    = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', prediction_data=True)
vectorizer    = CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=3)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    nr_topics='auto',
    calculate_probabilities=True,
    verbose=True
)

print('Entrenando BERTopic...')
topics, probs = topic_model.fit_transform(docs)
df_sector['topic'] = topics
n_t = topic_model.get_topic_info()['Topic'].nunique()
print(f'Topics encontrados: {n_t}')
topic_model.get_topic_info().head(15)

In [ ]:
# Ver palabras clave de cada topic para etiquetarlos manualmente
topic_info = topic_model.get_topic_info()
for tid in topic_info['Topic'].values:
    if tid == -1: continue
    words = [w for w, _ in topic_model.get_topic(tid)[:8]]
    sep = ' | '
    print(f'Topic {tid}: {sep.join(words)}')

In [ ]:
# ACCION MANUAL: ajusta los nombres segun las palabras clave que veas arriba
TOPIC_LABELS = {
    0: 'Transferencias internacionales',
    1: 'Comisiones y tarifas',
    2: 'Atencion al cliente',
    3: 'Cuenta y tarjeta',
    4: 'Verificacion de identidad KYC',
    5: 'App y experiencia digital',
    -1: 'Sin topic outlier'
}
df_sector['topic_label'] = df_sector['topic'].map(TOPIC_LABELS).fillna('Otros')
print('Labels asignados')

### 5.1 Topics de Wise

In [ ]:
df_wise_topics = df_sector[df_sector['company'] == TARGET].copy()

wise_topic_counts = (df_wise_topics[df_wise_topics['topic'] != -1]
                     ['topic_label'].value_counts())

fig, ax = plt.subplots(figsize=(8, 4))
wise_topic_counts.plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
ax.set_title('Topics en resenas de Wise.com', fontsize=13, fontweight='bold')
ax.set_xlabel('Numero de resenas')
sns.despine()
plt.tight_layout()
plt.savefig('wise_topics.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Topics en la competencia

In [ ]:
topic_by_company = (df_sector[df_sector['topic'] != -1]
                    .groupby(['company', 'topic_label'])
                    .size().reset_index(name='count'))

pivot = topic_by_company.pivot_table(index='company', columns='topic_label',
                                      values='count', fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).mul(100).round(1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(pivot_pct, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% resenas'})
ax.set_title('Distribucion de Topics por empresa', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wise_topics_competencia.png', dpi=150, bbox_inches='tight')
plt.show()

## PASO 6 - Sentimiento por Topic en Wise

In [ ]:
wise_cross = (df_wise_topics[df_wise_topics['topic'] != -1]
              .groupby(['topic_label', 'sentiment'])
              .size().reset_index(name='count'))

pivot_wise = wise_cross.pivot_table(index='topic_label', columns='sentiment',
                                     values='count', fill_value=0)
for col in ['Positivo', 'Neutro', 'Negativo']:
    if col not in pivot_wise.columns:
        pivot_wise[col] = 0
pivot_wise = pivot_wise[['Positivo', 'Neutro', 'Negativo']]
pivot_wise_pct = pivot_wise.div(pivot_wise.sum(axis=1), axis=0).mul(100)

fig, ax = plt.subplots(figsize=(9, 5))
pivot_wise_pct.plot(kind='barh', stacked=True, ax=ax,
                    color=[PALETTE['Positivo'], PALETTE['Neutro'], PALETTE['Negativo']],
                    edgecolor='white')
ax.set_title('Sentimiento por Topic - Wise.com', fontsize=13, fontweight='bold')
ax.set_xlabel('% resenas')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(title='Sentimiento', bbox_to_anchor=(1.02, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.savefig('wise_sentimiento_por_topic.png', dpi=150, bbox_inches='tight')
plt.show()

## PASO 7 - Sentimiento por Topic: Wise vs Competencia

In [ ]:
def pct_positive(group):
    return (group['sentiment'] == 'Positivo').mean() * 100

sent_by_topic_company = (df_sector[df_sector['topic'] != -1]
                         .groupby(['company', 'topic_label'])
                         .apply(pct_positive)
                         .reset_index(name='pct_positive'))

sector_avg = (sent_by_topic_company
              .groupby('topic_label')['pct_positive']
              .mean().reset_index(name='sector_avg'))
wise_avg = (sent_by_topic_company[sent_by_topic_company['company'] == TARGET]
            [['topic_label', 'pct_positive']]
            .rename(columns={'pct_positive': 'wise_pct'}))
compare = sector_avg.merge(wise_avg, on='topic_label', how='left').fillna(0)
compare['gap'] = compare['wise_pct'] - compare['sector_avg']
compare = compare.sort_values('gap', ascending=True)
print(compare.to_string(index=False))

In [ ]:
colors_gap = ['#2ecc71' if g >= 0 else '#e74c3c' for g in compare['gap']]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(compare['topic_label'], compare['gap'], color=colors_gap, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Wise vs Media del sector - Gap sentimiento positivo por topic',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Gap (pp)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
sns.despine()
plt.tight_layout()
plt.savefig('wise_gap_competencia_por_topic.png', dpi=150, bbox_inches='tight')
plt.show()

## PASO 8 - Conclusiones y areas de mejora

In [ ]:
print('=' * 60)
print('CONCLUSIONES - WISE.COM')
print('=' * 60)

dist = df_wise_topics['sentiment'].value_counts(normalize=True).mul(100).round(1)
print('1. Sentimiento global:')
for label in ['Positivo', 'Neutro', 'Negativo']:
    val = dist.get(label, 0)
    print(f'   {label}: {val}%')

wise_rank = pos_comp.reset_index(drop=True)
wise_rows = wise_rank[wise_rank['company'] == TARGET].index
if len(wise_rows):
    pos = wise_rows[0] + 1
    tot = len(wise_rank)
    print(f'2. Wise ocupa el puesto {pos}/{tot} en resenas positivas del sector.')

weak = compare[compare['gap'] < 0].sort_values('gap')
print('3. Areas de mejora (peor que la media del sector):')
for _, row in weak.iterrows():
    tl = row['topic_label']
    wp = row['wise_pct']
    sa = row['sector_avg']
    g  = row['gap']
    print(f'   - {tl}: Wise {wp:.1f}% vs Sector {sa:.1f}% (gap {g:.1f}pp)')

strong = compare[compare['gap'] >= 0].sort_values('gap', ascending=False)
print('4. Ventajas competitivas (mejor que la media del sector):')
for _, row in strong.iterrows():
    tl = row['topic_label']
    wp = row['wise_pct']
    sa = row['sector_avg']
    g  = row['gap']
    print(f'   + {tl}: Wise {wp:.1f}% vs Sector {sa:.1f}% (+{g:.1f}pp)')

print('=' * 60)

In [ ]:
try:
    from wordcloud import WordCloud
    neg_text = ' '.join(df_wise_topics[df_wise_topics['sentiment'] == 'Negativo']['text_clean'].tolist())
    wc = WordCloud(width=800, height=400, background_color='white',
                   colormap='Reds', max_words=80).generate(neg_text)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Palabras mas frecuentes en resenas NEGATIVAS - Wise.com',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('wise_wordcloud_negativo.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print('pip install wordcloud')

---
## Outputs generados

| Archivo | Contenido | Slide |
|---|---|---|
| wise_sentimiento_global.png | % positivo/neutro/negativo en Wise | Slide 3 |
| wise_sentimiento_competencia.png | Ranking sector | Slide 3 |
| wise_topics.png | Topics en Wise | Slide 4 |
| wise_topics_competencia.png | Heatmap topics por empresa | Slide 4 |
| wise_sentimiento_por_topic.png | Sentimiento por topic en Wise | Slide 4 |
| wise_gap_competencia_por_topic.png | Wise vs sector por topic | Slide 4 |
| wise_wordcloud_negativo.png | Palabras clave resenas negativas | Slide 5 |